# 🚀 Pipeline ETL com IA Generativa
### DIO TOTVS 2026 | Fundamentos de Engenharia de Dados e Machine Learning

---

## 📋 Visão Geral

Este notebook implementa um **Pipeline ETL completo** que usa **IA Generativa (Claude)** para enriquecer dados de clientes bancários com mensagens personalizadas.

```
┌─────────────┐     ┌──────────────────────────┐     ┌─────────────────────┐
│  EXTRACT    │────►│       TRANSFORM          │────►│       LOAD          │
│             │     │                          │     │                     │
│  CSV com    │     │  1. Classificar perfil   │     │  JSON enriquecido   │
│  usuários   │     │  2. IA Generativa gera   │     │  CSV para BI/DW     │
│  bancários  │     │     mensagem pessoal     │     │                     │
└─────────────┘     └──────────────────────────┘     └─────────────────────┘
```

### Tecnologias
| Tecnologia | Uso |
|-----------|-----|
| `pandas`    | Leitura e manipulação do CSV |
| `anthropic` | SDK para Claude (IA Generativa) |
| `dataclasses` | Modelagem de dados tipada |
| `python-dotenv` | Gerenciamento seguro de credenciais |

## 0️⃣ Instalação das Dependências

In [ ]:
# Execute apenas na primeira vez
%pip install anthropic pandas python-dotenv --quiet

## 0️⃣ Imports e Configuração

In [ ]:
import json
import os
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional

import anthropic
import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # Lê o arquivo .env

# ─── Configurações do pipeline ───────────────────────────────────────────────
INPUT_PATH  = Path("data/usuarios.csv")
OUTPUT_JSON = Path("output/usuarios_enriquecidos.json")
OUTPUT_CSV  = Path("output/usuarios_enriquecidos.csv")
MODEL       = "claude-sonnet-4-20250514"
MAX_TOKENS  = 300

print("✅ Imports OK")
print(f"📂 Input  : {INPUT_PATH}")
print(f"📤 Output : {OUTPUT_JSON} | {OUTPUT_CSV}")

## 📐 Modelo de Dados

In [ ]:
@dataclass
class Usuario:
    """Representa um cliente bancário com seus dados financeiros e
    o conteúdo gerado pela IA durante a etapa de transformação."""
    id: int
    nome: str
    conta: str
    cartao: str
    saldo: float
    limite_credito: float
    mensagem_ia: Optional[str] = None
    perfil_financeiro: Optional[str] = None
    status_processamento: str = "pendente"

print("✅ Modelo de dados definido")

---
## 1️⃣ EXTRACT — Extração dos Dados

> **Objetivo:** Ler o CSV de entrada e converter para objetos Python tipados.
> Em produção, essa etapa poderia ser uma chamada a uma API REST, uma consulta a banco de dados, ou leitura de um Data Lake.

In [ ]:
def extract(caminho: Path) -> list[Usuario]:
    """Extrai usuários do CSV e valida colunas obrigatórias."""
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")

    df = pd.read_csv(caminho)

    # Validação de schema
    colunas_obrigatorias = {"id", "nome", "conta", "cartao", "saldo", "limite_credito"}
    faltando = colunas_obrigatorias - set(df.columns)
    if faltando:
        raise ValueError(f"Colunas ausentes no CSV: {faltando}")

    return [
        Usuario(
            id=int(row["id"]),
            nome=str(row["nome"]),
            conta=str(row["conta"]),
            cartao=str(row["cartao"]),
            saldo=float(row["saldo"]),
            limite_credito=float(row["limite_credito"]),
        )
        for _, row in df.iterrows()
    ]


# ─── Executar Extract ─────────────────────────────────────────────────────────
usuarios = extract(INPUT_PATH)

print(f"✅ {len(usuarios)} usuário(s) extraído(s)\n")

# Visualizar como DataFrame
pd.DataFrame([asdict(u) for u in usuarios])[
    ["id", "nome", "conta", "cartao", "saldo", "limite_credito"]
]

---
## 2️⃣ TRANSFORM — Enriquecimento com IA Generativa

> **Objetivo:** Enriquecer cada registro com:
> - **Perfil financeiro** — classificado por regra de negócio local (rápido, sem I/O).
> - **Mensagem personalizada** — gerada pela IA Claude com base no perfil do cliente.
>
> ⚠️ **Pré-requisito:** defina a variável `ANTHROPIC_API_KEY` no arquivo `.env`.

In [ ]:
def _classificar_perfil(saldo: float, limite: float) -> str:
    """Regra de negócio: classifica o cliente por saúde financeira."""
    utilizacao = (limite - saldo) / limite if limite > 0 else 0
    if saldo > 4000 and utilizacao < 0.3:
        return "Premium"
    elif saldo > 1000:
        return "Intermediário"
    return "Iniciante"


def _gerar_mensagem_ia(cliente: anthropic.Anthropic, usuario: Usuario) -> str:
    """Chama a API da IA Generativa para criar mensagem personalizada."""
    prompt = f"""Você é um assistente financeiro do TOTVS / Banco de Dados.
Gere uma mensagem curta (máx. 2 frases), personalizada e motivadora para o seguinte cliente:

- Nome: {usuario.nome}
- Perfil financeiro: {usuario.perfil_financeiro}
- Saldo atual: R$ {usuario.saldo:,.2f}
- Limite de crédito: R$ {usuario.limite_credito:,.2f}

A mensagem deve ser direta, amigável e incluir uma dica ou incentivo financeiro relevante ao perfil.
Responda APENAS com a mensagem, sem saudação formal nem assinatura."""

    resposta = cliente.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        messages=[{"role": "user", "content": prompt}],
    )
    return resposta.content[0].text.strip()


def transform(usuarios: list[Usuario]) -> list[Usuario]:
    """Transforma os usuários enriquecendo com perfil e mensagem de IA."""
    api_key = os.getenv("ANTHROPIC_API_KEY")
    if not api_key:
        raise EnvironmentError(
            "ANTHROPIC_API_KEY não encontrada. Configure o arquivo .env!"
        )

    cliente = anthropic.Anthropic(api_key=api_key)

    for i, usuario in enumerate(usuarios, 1):
        print(f"[{i}/{len(usuarios)}] {usuario.nome}")
        try:
            usuario.perfil_financeiro = _classificar_perfil(
                usuario.saldo, usuario.limite_credito
            )
            print(f"  ↳ Perfil  : {usuario.perfil_financeiro}")

            usuario.mensagem_ia = _gerar_mensagem_ia(cliente, usuario)
            print(f"  ↳ Mensagem: {usuario.mensagem_ia}")

            usuario.status_processamento = "sucesso"

        except anthropic.RateLimitError:
            print("  ↳ ⚠️  Rate limit. Aguardando 10s...")
            time.sleep(10)
            usuario.mensagem_ia = "[pausado — rate limit]"
            usuario.status_processamento = "rate_limit"

        except Exception as exc:
            print(f"  ↳ ❌ Erro: {exc}")
            usuario.mensagem_ia = "[erro na geração]"
            usuario.status_processamento = f"erro: {exc}"

        time.sleep(1)  # Respeitar rate limit da API

    return usuarios


# ─── Executar Transform ───────────────────────────────────────────────────────
usuarios = transform(usuarios)

sucessos = sum(1 for u in usuarios if u.status_processamento == "sucesso")
print(f"\n✅ Transform concluído: {sucessos}/{len(usuarios)} com sucesso")

---
## 3️⃣ LOAD — Persistência dos Dados Enriquecidos

> **Objetivo:** Salvar os dados transformados em dois formatos:
> - **JSON** — ideal para consumo por APIs e microserviços.
> - **CSV** — ideal para ingestão em bancos de dados, DW ou ferramentas de BI.

In [ ]:
def load(usuarios: list[Usuario]) -> None:
    """Persiste os dados enriquecidos em JSON e CSV."""
    OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
    registros = [asdict(u) for u in usuarios]

    # JSON
    with OUTPUT_JSON.open("w", encoding="utf-8") as f:
        json.dump(registros, f, ensure_ascii=False, indent=2)
    print(f"✅ JSON → {OUTPUT_JSON}")

    # CSV (utf-8-sig garante compatibilidade com Excel)
    pd.DataFrame(registros).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"✅ CSV  → {OUTPUT_CSV}")


# ─── Executar Load ────────────────────────────────────────────────────────────
load(usuarios)

# Preview do resultado
print("\n📊 Preview dos dados enriquecidos:")
pd.DataFrame([asdict(u) for u in usuarios])[
    ["nome", "perfil_financeiro", "mensagem_ia", "status_processamento"]
]

---
## ✅ Pipeline Completo — Execução em Uma Célula

> Se quiser rodar tudo de uma vez, execute a célula abaixo:

In [ ]:
if __name__ == "__main__":
    print("=" * 60)
    print("  Pipeline ETL — DIO TOTVS 2026 | Engenharia de Dados e ML")
    print("=" * 60)
    start = time.perf_counter()

    usuarios = extract(INPUT_PATH)
    usuarios = transform(usuarios)
    load(usuarios)

    print(f"\n🏁 Concluído em {time.perf_counter() - start:.2f}s")

---
## 📊 Análise dos Resultados

In [ ]:
# Lê o JSON de saída e exibe estatísticas
with OUTPUT_JSON.open(encoding="utf-8") as f:
    dados = json.load(f)

df_resultado = pd.DataFrame(dados)

print("📈 Distribuição de Perfis:")
print(df_resultado["perfil_financeiro"].value_counts().to_string())

print("\n💰 Estatísticas de Saldo por Perfil:")
print(
    df_resultado.groupby("perfil_financeiro")["saldo"]
    .agg(["mean", "min", "max"])
    .round(2)
    .to_string()
)

print("\n📬 Mensagens Geradas:")
for _, row in df_resultado.iterrows():
    print(f"  [{row['perfil_financeiro']:13s}] {row['nome']:25s} → {row['mensagem_ia']}")

---
## 🧠 Conceitos Aprendidos

| Etapa | O que fizemos | Por que importa |
|-------|--------------|------------------|
| **Extract** | Lemos um CSV com validação de schema | Dados ruins entram → dados ruins saem |
| **Transform** | Classificamos por regra + chamamos IA | O valor do ETL está na transformação |
| **Load** | Salvamos em JSON e CSV | Formatos diferentes para consumos diferentes |

### Boas práticas implementadas
- ✅ **Tipagem forte** com `dataclasses` — evita bugs silenciosos
- ✅ **Tratamento de erros** por usuário — pipeline não para se um falhar
- ✅ **Rate limit handling** — respeita limites da API
- ✅ **Separação de responsabilidades** — cada função tem uma única responsabilidade
- ✅ **Credenciais via `.env`** — nunca em código fonte
- ✅ **Logging estruturado** — rastreabilidade do pipeline